# Практика · Panoptic і SAM: одна відповідь на кожен піксель

> Лекція: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) ·
> Домашнє завдання: [homework.html](homework.html)

> 🔌 **Мережа не потрібна.** Усі сцени зошит малює формулами. Досить `torch`,
> `numpy`, `opencv-python` і `matplotlib`. Ваг справжнього SAM тут немає й не буде:
> пакета `segment_anything` у середовищі не встановлено, а тягнути 2.4 ГБ ваг з
> мережі зошит не має права. Тому ми **навчимо власний крихітний сегментатор за
> підказкою** — так само, як [тема 22](../22-clip/lecture.html) зробила з CLIP, а
> [тема 25](../25-yolo/lecture.html) із YOLO.

> ⏱ Зошит навчає **21 мережу**: по три зерна на сім настройок. Заміряно: сам зошит
> друкує в кінці **340 секунд**, а перевірка разом із запуском ядра — **414**. Тобто
> близько шести хвилин на чотирьох ядрах без відеокарти, в один потік. Майже весь
> цей час — навчання; решта зошита йде за секунди, і в кінці друкується розклад
> часу по групах мереж.

[Тема 31](../31-instance-segmentation/lecture.html) закінчилась розривом: у нас є
семантична мережа, яка добре фарбує фон, і instance-мережа, яка добре знаходить
предмети, — але це **дві різні моделі на одне зображення**. Що зробимо:

1. зшиємо їхні відповіді й порахуємо **конфлікти й дірки** — головне число теми;
2. напишемо **свою PQ** з нуля, звіримо на прикладі, порахованому руками, і
   розкладемо її на **SQ** і **RQ**;
3. подивимось, де PQ впорядковує моделі **інакше**, ніж пара mIoU + mask AP;
4. розділимо PQ на **речі** й **матерію** і побачимо, чому середнє по них оманливе;
5. зберемо **власну panoptic-голову** — семантична гілка плюс голосування за центр;
6. навчимо мережу, яка приймає зображення **і точку**, а віддає маску;
7. поміряємо, скільки коштує **одна маска замість трьох** на неоднозначну підказку;
8. порівняємо підказку **точкою й рамкою**.

Усе — на трьох зернах, бо різниця, менша за розкид, різницею не є.

In [ ]:
import math
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import cv2
import matplotlib.pyplot as plt
from torchvision.ops import nms, roi_align

# один потік дає і швидкість на маленьких тензорах, і повторюваність чисел:
# під кількома потоками float-суми йдуть в іншому порядку
torch.set_num_threads(1)

TRAIN_SCENES, TEST_SCENES = 130, 70       # скільки сцен малюємо
SEEDS = (0, 1, 2)                         # три зерна на кожну точку
SEM_WIDTH, SEM_EPOCHS = 10, 6             # семантична мережа
DET_EPOCHS, DET_BATCH = 12, 16            # instance-мережа з теми 31
PAN_WIDTH, PAN_EPOCHS = 12, 6             # власна panoptic-голова
PROMPT_WIDTH, PROMPT_EPOCHS = 8, 5        # мережа за підказкою
AMBIGUOUS_EPOCHS = 3                # неоднозначні підказки: даних утричі більше

NOTEBOOK_STARTED = time.time()
print("torch   ", torch.__version__)
print("cv2     ", cv2.__version__)
print("numpy   ", np.__version__)
print("потоків ", torch.get_num_threads())

## 1 · Сцени: генератор із теми 28, без жодної зміни

Правило блоку: **генератор беремо готовий**. Варто змінити в ньому одну дрібницю —
і попливуть частки класів, а з ними всі числа блоку. Тому нижче стоїть той самий
код, що в [темі 28](../28-detection-practice/lecture.html) і
[темі 31](../31-instance-segmentation/lecture.html): фігура кидається на полотно
64×64, нова не перекриває стару більш ніж на 45 % своєї площі, шум 0.12.

Додано рівно одне, і воно потрібне саме panoptic: разом із масками кожного предмета
ми одразу будуємо **panoptic-розмітку** — пару карт «клас пікселя» і «номер
предмета». Фон у цій парі отримує клас 0 і номер 0: він **матерія**, його не
рахують поштучно.

In [ ]:
SIZE = 64                                   # сторона полотна в пікселях
CLASS_NAMES = ["коло", "квадрат", "трикутник"]
CLASS_COUNT = 3
NUM_LABELS = CLASS_COUNT + 1                # три речі плюс фон-матерія
GRID = 8                                    # карта ознак 8×8
STRIDE = SIZE / GRID
ROWS, COLUMNS = np.mgrid[0:SIZE, 0:SIZE]


def shape_mask(kind, center_x, center_y, radius):
    """Маска однієї фігури на полотні 64×64 — та сама геометрія, що в блоці 5."""
    ys, xs = np.mgrid[0:SIZE, 0:SIZE]
    if kind == 0:                                    # коло
        return (xs - center_x) ** 2 + (ys - center_y) ** 2 <= radius * radius
    if kind == 1:                                    # квадрат
        return (np.abs(xs - center_x) <= radius) & (np.abs(ys - center_y) <= radius)
    # трикутник: ширина росте згори вниз
    return ((ys - center_y + radius >= 0) & (ys - center_y <= radius)
            & (np.abs(xs - center_x) <= (ys - center_y + radius) / 2.0))


def box_from_mask(mask):
    """Рамка з маски: край + 1 по правому й нижньому боці, як в угоді COCO."""
    ys, xs = np.nonzero(mask)
    return [float(xs.min()), float(ys.min()), float(xs.max() + 1), float(ys.max() + 1)]


def make_scene(rng, max_objects=3, overlap_limit=0.45):
    """Канонічна сцена теми 28 плюс маска кожної фігури окремо."""
    image = np.zeros((SIZE, SIZE), np.float32)
    boxes, labels, masks = [], [], []
    for _slot in range(int(rng.integers(1, max_objects + 1))):
        for _attempt in range(40):
            radius = int(rng.integers(6, 11))
            center_x = int(rng.integers(radius + 1, SIZE - radius - 1))
            center_y = int(rng.integers(radius + 1, SIZE - radius - 1))
            kind = int(rng.integers(0, 3))
            mask = shape_mask(kind, center_x, center_y, radius)
            box = box_from_mask(mask)

            # канонічне правило блоку: нова фігура не перекриває стару
            # більше ніж на 45 % своєї площі
            too_close = False
            for previous in boxes:
                width = max(0, min(box[2], previous[2]) - max(box[0], previous[0]))
                height = max(0, min(box[3], previous[3]) - max(box[1], previous[1]))
                if width * height > overlap_limit * (box[2] - box[0]) * (box[3] - box[1]):
                    too_close = True
                    break
            if too_close:
                continue

            image[mask] = 1.0
            boxes.append(box)
            labels.append(kind)
            masks.append(mask)
            break
    image = np.clip(image + rng.normal(0, 0.12, (SIZE, SIZE)).astype(np.float32), 0, 1)
    return (image, np.array(boxes, np.float32).reshape(-1, 4),
            np.array(labels, np.int64), masks)


def visible_masks(masks):
    """Пізніші фігури затуляють ранніх — лишаємо тільки видимі пікселі."""
    out = []
    for index, mask in enumerate(masks):
        visible = mask.copy()
        for later in masks[index + 1:]:
            visible &= ~later
        out.append(visible)
    return out


def make_dataset(seed, count, max_objects=6):
    """Набір сцен: картинка, видимі маски, рамки з них, семантична й panoptic карти."""
    rng = np.random.default_rng(seed)
    out = []
    for _ in range(count):
        image, _boxes, labels, masks = make_scene(rng, max_objects)
        visible = visible_masks(masks)
        # предмет, затулений цілком, із розмітки зникає — його просто не видно
        keep = [i for i in range(len(visible)) if visible[i].sum() > 0]
        visible = [visible[i] for i in keep]
        labels = labels[keep]
        boxes = np.array([box_from_mask(m) for m in visible], np.float32).reshape(-1, 4)
        segmentation = np.zeros((SIZE, SIZE), np.int64)
        instance_map = np.zeros((SIZE, SIZE), np.int64)
        for index, mask in enumerate(visible):
            segmentation[mask] = labels[index] + 1
            instance_map[mask] = index + 1      # фон лишається нулем: він матерія
        out.append(dict(image=image, masks=visible, labels=labels, boxes=boxes,
                        seg=segmentation, ids=instance_map))
    return out


started = time.time()
train_data = make_dataset(42, TRAIN_SCENES)      # щільні сцени: до 6 предметів
test_data = make_dataset(7, TEST_SCENES)
print("згенеровано за %.2f с" % (time.time() - started))
for name, data in (("навчальні", train_data), ("перевірні", test_data)):
    total = sum(len(scene["labels"]) for scene in data)
    print("  %-10s сцен %3d, предметів %3d, у середньому %.2f на сцену"
          % (name, len(data), total, total / len(data)))

### Звірка: чи той самий це генератор

Найдешевша перевірка — частки класів. [Тема 29](../29-semantic-segmentation/lecture.html)
на канонічних сценах (до 3 предметів) дістала фон **89.34 %** на навчальній частині,
і саме це число потім стало мотивом усього блоку. Наші сцени щільніші, тож фону
має бути **менше**, а порядок класів — той самий: квадрат більший за коло, бо
квадрат зі стороною `2r` займає ≈ `4r²`, а коло — ≈ `3.14 r²`.

In [ ]:
def class_shares(data):
    """Скільки відсотків пікселів припадає на кожен клас."""
    counts = np.zeros(NUM_LABELS, np.int64)
    for scene in data:
        counts += np.bincount(scene["seg"].ravel(), minlength=NUM_LABELS)
    return 100.0 * counts / counts.sum()


names = ["фон"] + CLASS_NAMES
print("  клас        | навчальні | перевірні")
train_shares, test_shares = class_shares(train_data), class_shares(test_data)
for index in range(NUM_LABELS):
    print("  %-11s |  %6.2f %% |  %6.2f %%"
          % (names[index], train_shares[index], test_shares[index]))
print()
assert train_shares[2] > train_shares[1], "квадрат мусить бути більшим за коло!"
print("✅ квадрат більший за коло — геометрія та сама, що в темі 29")
print("Фону менше за 89.34 % теми 29 рівно тому, що сцени щільніші:")
print("до шести предметів замість трьох.")

## 2 · Дві моделі на одне зображення

Перша модель — **семантична**: маленький U-Net на два рівні стискання зі скіпами
([тема 30](../30-unet/lecture.html)), який кожному пікселю призначає один із
чотирьох класів. Вона знає про фон і нічого не знає про окремі предмети.

Друга модель — **instance**: anchor-free детектор із гілкою маски, узятий із
[теми 31](../31-instance-segmentation/lecture.html) без змін. Вона знає про окремі
предмети й нічого не знає про фон: її вихід — список масок, а що між ними, її не
обходить.

Це дві **різні** мережі з різними вагами. Саме тому їхні відповіді не зобовʼязані
узгоджуватись — і зараз ми порахуємо, наскільки саме вони не узгоджуються.

In [ ]:
def unet_block(in_channels, out_channels):
    """Дві згортки 3×3 поспіль — стандартна цеглинка U-Net."""
    return nn.Sequential(nn.Conv2d(in_channels, out_channels, 3, padding=1),
                         nn.BatchNorm2d(out_channels), nn.ReLU(),
                         nn.Conv2d(out_channels, out_channels, 3, padding=1),
                         nn.BatchNorm2d(out_channels), nn.ReLU())


class UNetBody(nn.Module):
    """Тіло на два рівні стискання зі скіпами — спільне для трьох мереж зошита."""

    def __init__(self, in_channels=1, width=16):
        super().__init__()
        self.down1 = unet_block(in_channels, width)
        self.down2 = unet_block(width, width * 2)
        self.bottom = unet_block(width * 2, width * 4)
        self.up2 = unet_block(width * 4 + width * 2, width * 2)
        self.up1 = unet_block(width * 2 + width, width)

    def forward(self, x):
        first = self.down1(x)                                   # 64×64
        second = self.down2(F.max_pool2d(first, 2))             # 32×32
        deep = self.bottom(F.max_pool2d(second, 2))             # 16×16
        deep = F.interpolate(deep, scale_factor=2, mode="nearest")
        second = self.up2(torch.cat([deep, second], dim=1))     # скіп із 32×32
        second = F.interpolate(second, scale_factor=2, mode="nearest")
        return self.up1(torch.cat([second, first], dim=1))      # скіп із 64×64


class SemanticNet(nn.Module):
    """Модель теми 29: клас кожного пікселя і жодного поняття про предмети."""

    def __init__(self, width=SEM_WIDTH):
        super().__init__()
        self.body = UNetBody(1, width)
        self.semantic = nn.Conv2d(width, NUM_LABELS, 1)

    def forward(self, x):
        return self.semantic(self.body(x))


def train_semantic(data, seed, epochs=SEM_EPOCHS, batch_size=8, learning_rate=3e-3):
    torch.manual_seed(seed)
    model = SemanticNet()
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    images = torch.from_numpy(np.stack([s["image"] for s in data])[:, None])
    target = torch.from_numpy(np.stack([s["seg"] for s in data]))
    started = time.time()
    model.train()
    for _epoch in range(epochs):
        order = torch.randperm(len(images))
        for start in range(0, len(images), batch_size):
            batch = order[start:start + batch_size]
            loss = F.cross_entropy(model(images[batch]), target[batch])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    return model, time.time() - started


@torch.no_grad()
def predict_semantic(model, data):
    """Карта класів для кожної сцени — саме те, що вміє віддати тема 29."""
    model.eval()
    images = torch.from_numpy(np.stack([s["image"] for s in data])[:, None])
    pieces = []
    for index in range(0, len(images), 16):
        pieces.append(model(images[index:index + 16]).argmax(dim=1).numpy())
    return np.concatenate(pieces)


print("параметрів у семантичній мережі: %d" % sum(p.numel() for p in SemanticNet().parameters()))

### Instance-модель: код теми 31 без правок

Нижче — та сама голова, що в темі 31: карта 8×8, три числа класу в кожній позиції,
чотири відстані до країв рамки, одне число `centerness`, і гілка маски, яка вирізає
ділянку 14×14 з проміжної карти й розтягує її до 28×28. Дивитись на неї уважно тут
не треба — вона вже розібрана в темі 31. Нам важливий лише її **вихід**: список
масок із класами й упевненостями.

In [ ]:
POINTS = torch.tensor([[(col + 0.5) * STRIDE, (row + 0.5) * STRIDE]
                       for row in range(GRID) for col in range(GRID)],
                      dtype=torch.float32)
PRIOR = 0.01                                   # бажана ймовірність предмета на старті
BIAS_INIT = -math.log((1 - PRIOR) / PRIOR)
MASK_SIDE = 28
FEATURE_SCALE = 0.25                           # карта 16×16 відносно входу 64×64


def focal_loss(logits, targets, alpha=0.25, gamma=2.0):
    """Стійка версія: крос-ентропію рахуємо з логітів, не переходячи через p."""
    probability = torch.sigmoid(logits)
    cross_entropy = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    p_t = probability * targets + (1 - probability) * (1 - targets)
    alpha_t = alpha * targets + (1 - alpha) * (1 - targets)
    return alpha_t * cross_entropy * (1 - p_t) ** gamma


def free_targets(boxes, labels):
    """Ціль anchor-free: позиція позитивна, якщо вона всередині рамки."""
    total = POINTS.shape[0]
    positive = torch.zeros(total, dtype=torch.bool)
    class_id = torch.full((total,), -1, dtype=torch.long)
    distances = torch.zeros(total, 4)
    if len(boxes) == 0:
        return positive, class_id, distances
    truth = torch.as_tensor(np.ascontiguousarray(boxes), dtype=torch.float32)
    areas = (truth[:, 2] - truth[:, 0]) * (truth[:, 3] - truth[:, 1])
    point_x, point_y = POINTS[:, 0:1], POINTS[:, 1:2]
    left, top = point_x - truth[:, 0], point_y - truth[:, 1]
    right, bottom = truth[:, 2] - point_x, truth[:, 3] - point_y
    inside = (left > 0) & (top > 0) & (right > 0) & (bottom > 0)
    area_or_infinity = torch.where(inside, areas.expand_as(inside),
                                   torch.full_like(inside, float("inf"),
                                                   dtype=torch.float32))
    smallest, chosen = area_or_infinity.min(dim=1)
    positive = torch.isfinite(smallest)
    index = torch.arange(total)
    stacked = torch.stack([left[index, chosen], top[index, chosen],
                           right[index, chosen], bottom[index, chosen]], dim=1)
    distances[positive] = stacked[positive] / STRIDE
    class_id[positive] = torch.as_tensor(labels, dtype=torch.long)[chosen[positive]]
    return positive, class_id, distances


def centerness_target(distances):
    """Наскільки позиція близька до центра рамки: 1 у центрі, 0 на краю."""
    left, top, right, bottom = distances.unbind(dim=1)
    horizontal = torch.min(left, right) / torch.max(left, right).clamp(min=1e-6)
    vertical = torch.min(top, bottom) / torch.max(top, bottom).clamp(min=1e-6)
    return torch.sqrt((horizontal * vertical).clamp(min=0))


class DetectorBody(nn.Module):
    """Тіло теми 26; додатково віддає карту 16×16 для гілки маски."""

    def __init__(self):
        super().__init__()

        def block(in_channels, out_channels):
            return nn.Sequential(nn.Conv2d(in_channels, out_channels, 3, padding=1),
                                 nn.BatchNorm2d(out_channels), nn.ReLU(), nn.MaxPool2d(2))

        self.stage1 = block(1, 16)
        self.stage2 = block(16, 32)
        self.stage3 = block(32, 64)
        self.tail = nn.Sequential(nn.Conv2d(64, 64, 3, padding=1),
                                  nn.BatchNorm2d(64), nn.ReLU())

    def forward(self, x):
        middle = self.stage2(self.stage1(x))            # 16×16, 32 канали
        return self.tail(self.stage3(middle)), middle


class MaskDetector(nn.Module):
    """Детектор із гілкою маски — Mask R-CNN у мініатюрі, тема 31."""

    def __init__(self):
        super().__init__()
        self.body = DetectorBody()
        self.classifier = nn.Conv2d(64, CLASS_COUNT, 3, padding=1)
        self.regressor = nn.Conv2d(64, 4, 3, padding=1)
        self.centerness = nn.Conv2d(64, 1, 3, padding=1)
        nn.init.constant_(self.classifier.bias, BIAS_INIT)
        self.mask_head = nn.Sequential(
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(),
            nn.ConvTranspose2d(32, 32, 2, stride=2), nn.ReLU(),
            nn.Conv2d(32, CLASS_COUNT, 1))

    def forward(self, x):
        features, middle = self.body(x)
        count = x.shape[0]
        logits = self.classifier(features).permute(0, 2, 3, 1).reshape(count, -1, CLASS_COUNT)
        distances = F.relu(self.regressor(features)).permute(0, 2, 3, 1).reshape(count, -1, 4)
        center = self.centerness(features).permute(0, 2, 3, 1).reshape(count, -1)
        return logits, distances, center, middle

    def masks_for(self, middle, rois):
        patches = roi_align(middle, rois, output_size=(14, 14),
                            spatial_scale=FEATURE_SCALE, sampling_ratio=2, aligned=True)
        return self.mask_head(patches)


print("параметрів в instance-мережі: %d" % sum(p.numel() for p in MaskDetector().parameters()))

In [ ]:
def mask_targets(masks, boxes, side=MASK_SIDE):
    """Істинна маска, вирізана за своєю рамкою й стиснута до side×side."""
    out = []
    for mask, box in zip(masks, boxes):
        x0, y0 = int(box[0]), int(box[1])
        x1, y1 = int(np.ceil(box[2])), int(np.ceil(box[3]))
        crop = torch.from_numpy(mask[y0:y1, x0:x1].astype(np.float32))[None, None]
        out.append(F.interpolate(crop, size=(side, side), mode="bilinear",
                                 align_corners=False)[0, 0])
    if not out:
        return torch.zeros(0, side, side)
    return torch.stack(out)


def pack_detector(data):
    """Готуємо цілі один раз, щоб не рахувати їх у кожній епосі."""
    images = torch.from_numpy(np.stack([scene["image"] for scene in data])[:, None])
    total = POINTS.shape[0]
    positive = torch.zeros(len(data), total, dtype=torch.bool)
    class_target = torch.zeros(len(data), total, CLASS_COUNT)
    distance_target = torch.zeros(len(data), total, 4)
    center_target = torch.zeros(len(data), total)
    gt_boxes, gt_masks, gt_labels = [], [], []
    for index, scene in enumerate(data):
        is_positive, class_id, distances = free_targets(scene["boxes"], scene["labels"])
        positive[index] = is_positive
        if is_positive.any():
            class_target[index, is_positive, class_id[is_positive]] = 1.0
            distance_target[index, is_positive] = distances[is_positive]
            center_target[index, is_positive] = centerness_target(distances[is_positive])
        gt_boxes.append(torch.as_tensor(scene["boxes"], dtype=torch.float32))
        gt_labels.append(torch.as_tensor(scene["labels"], dtype=torch.long))
        gt_masks.append(mask_targets(scene["masks"], scene["boxes"]))
    return dict(images=images, positive=positive, cls=class_target, dist=distance_target,
                center=center_target, gt_boxes=gt_boxes, gt_masks=gt_masks,
                gt_labels=gt_labels)


def paste_mask(small, box, threshold=0.5):
    """Маска 28×28 розтягується під розмір рамки й вклеюється в полотно."""
    x0, y0 = max(0, int(np.floor(box[0]))), max(0, int(np.floor(box[1])))
    x1, y1 = int(np.ceil(box[2])), int(np.ceil(box[3]))
    x1, y1 = min(SIZE, max(x0 + 1, x1)), min(SIZE, max(y0 + 1, y1))
    patch = torch.from_numpy(small.astype(np.float32))[None, None]
    back = F.interpolate(patch, size=(y1 - y0, x1 - x0), mode="bilinear",
                         align_corners=False)[0, 0].numpy()
    out = np.zeros((SIZE, SIZE), bool)
    out[y0:y1, x0:x1] = back >= threshold
    return out


def train_detector(data_pack, seed, epochs=DET_EPOCHS, batch_size=DET_BATCH,
                   learning_rate=3e-3):
    torch.manual_seed(seed)
    model = MaskDetector()
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    images = data_pack["images"]
    started = time.time()
    model.train()
    for _epoch in range(epochs):
        order = torch.randperm(len(images))
        for start in range(0, len(images), batch_size):
            batch = order[start:start + batch_size]
            logits, distances, center, middle = model(images[batch])
            positive = data_pack["positive"][batch]
            loss = focal_loss(logits, data_pack["cls"][batch]).sum()
            if positive.any():
                loss = loss + F.smooth_l1_loss(distances[positive],
                                               data_pack["dist"][batch][positive],
                                               reduction="sum")
                loss = loss + F.binary_cross_entropy_with_logits(
                    center[positive], data_pack["center"][batch][positive], reduction="sum")
            loss = loss / max(1, int(positive.sum().item()))

            # гілку маски вчимо на істинних рамках — так само, як у Mask R-CNN
            rois, targets, labels = [], [], []
            for slot, scene_index in enumerate(batch.tolist()):
                boxes = data_pack["gt_boxes"][scene_index]
                if len(boxes) == 0:
                    continue
                rois.append(torch.cat([torch.full((len(boxes), 1), float(slot)), boxes], dim=1))
                targets.append(data_pack["gt_masks"][scene_index])
                labels.append(data_pack["gt_labels"][scene_index])
            if rois:
                predicted = model.masks_for(middle, torch.cat(rois))
                labels = torch.cat(labels)
                predicted = predicted[torch.arange(len(labels)), labels]
                loss = loss + F.binary_cross_entropy_with_logits(predicted, torch.cat(targets))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    return model, time.time() - started


@torch.no_grad()
def predict_instances(model, data, score_floor=0.30, nms_threshold=0.5):
    """Список предметів: рамка, клас, упевненість і маска на повному полотні."""
    model.eval()
    images = torch.from_numpy(np.stack([scene["image"] for scene in data])[:, None])
    out = []
    for index in range(len(data)):
        logits, distances, center, middle = model(images[index:index + 1])
        probability = torch.sigmoid(logits[0])
        step = distances[0] * STRIDE
        boxes = torch.stack([POINTS[:, 0] - step[:, 0], POINTS[:, 1] - step[:, 1],
                             POINTS[:, 0] + step[:, 2], POINTS[:, 1] + step[:, 3]],
                            dim=1).clamp(0, SIZE)
        score, class_id = probability.max(dim=1)
        score = torch.sqrt(score * torch.sigmoid(center[0]))     # оцінка з centerness
        keep = (score >= score_floor) & (boxes[:, 2] - boxes[:, 0] > 1) \
            & (boxes[:, 3] - boxes[:, 1] > 1)
        boxes, score, class_id = boxes[keep], score[keep], class_id[keep]
        if len(boxes):
            keep = nms(boxes, score, nms_threshold)
            boxes, score, class_id = boxes[keep], score[keep], class_id[keep]
        masks = []
        if len(boxes):
            rois = torch.cat([torch.zeros(len(boxes), 1), boxes], dim=1)
            logit_maps = model.masks_for(middle, rois)
            for k in range(len(boxes)):
                channel = logit_maps[k, class_id[k]]
                masks.append(paste_mask(torch.sigmoid(channel).numpy(), boxes[k].numpy()))
        out.append(dict(boxes=boxes.numpy(), scores=score.numpy(),
                        labels=class_id.numpy(), masks=masks))
    return out


detector_pack = pack_detector(train_data)
print("цілі детектора готові: %d сцен, %d позицій на сцену"
      % (len(detector_pack["images"]), POINTS.shape[0]))

### Навчаємо обидві моделі, три зерна

Зерно керує і початковими вагами, і порядком прикладів. Пара «семантична модель
із зерном k» плюс «instance-модель із зерном k» — це і є та ситуація, яку в
реальному проєкті отримують, навчивши дві мережі окремо.

Це найдовша клітинка зошита.

In [ ]:
semantic_maps, instance_lists, semantic_time, instance_time = [], [], [], []
for seed in SEEDS:
    semantic_model, seconds_semantic = train_semantic(train_data, seed)
    detector_model, seconds_detector = train_detector(detector_pack, seed)
    semantic_maps.append(predict_semantic(semantic_model, test_data))
    instance_lists.append(predict_instances(detector_model, test_data))
    semantic_time.append(seconds_semantic)
    instance_time.append(seconds_detector)
    found = sum(len(p["masks"]) for p in instance_lists[-1])
    print("  зерно %d: семантична %3.0f с, instance %3.0f с, предметів видано %d"
          % (seed, seconds_semantic, seconds_detector, found))
truth_objects = sum(len(scene["masks"]) for scene in test_data)
print()
print("істинних предметів на перевірці: %d" % truth_objects)

## 2.1 · Замір №1: скільки пікселів дістали дві відповіді, а скільки — жодної

Кожен піксель дивиться на дві відповіді й може потрапити в одну з бід:

- **спірний** — обидві моделі щось сказали, але різне. Сюди ж піксель, який
  семантична модель назвала фоном, а instance-модель накрила маскою предмета;
- **перекритий** — його накрили **дві або більше** масок предметів. Instance-моделі
  це не заважає: її вихід і є список масок, які можуть перетинатись. Panoptic це
  забороняє;
- **дірка** — семантична модель назвала його предметом, а жодна маска його не
  накрила. Він лишається без номера предмета, тобто без відповіді.

Останні дві категорії — це саме те, чого не існує в постановці семантичної
сегментації й не існує в постановці instance. Вони зʼявляються **рівно від
зшивання**.

In [ ]:
def conflict_report(semantic_map, instances):
    """Скільки пікселів дістали дві різні відповіді, а скільки — жодної."""
    cover = np.zeros((SIZE, SIZE), np.int64)      # скільки масок накрило піксель
    top_class = np.zeros((SIZE, SIZE), np.int64)  # клас найупевненішої з них
    order = np.argsort(-instances["scores"]) if len(instances["masks"]) else []
    for k in order:
        mask = instances["masks"][k]
        cover += mask.astype(np.int64)
        fresh = mask & (top_class == 0)
        top_class[fresh] = int(instances["labels"][k]) + 1
    disagree = (cover >= 1) & (semantic_map > 0) & (top_class != semantic_map)
    only_instance = (cover >= 1) & (semantic_map == 0)
    overlap = cover >= 2
    hole = (cover == 0) & (semantic_map > 0)
    claimed = (cover >= 1) | (semantic_map > 0)   # піксель, який хоч хтось назвав предметом
    return dict(disagree=int(disagree.sum()), only_instance=int(only_instance.sum()),
                overlap=int(overlap.sum()), hole=int(hole.sum()),
                claimed=int(claimed.sum()),
                broken=int((disagree | only_instance | overlap | hole).sum()))


pixels_total = len(test_data) * SIZE * SIZE
conflict_rows = []
print("  зерно | спірних | перекритих | дірок | зіпсовано серед пікселів предметів")
for seed_index, seed in enumerate(SEEDS):
    add = dict(disagree=0, only_instance=0, overlap=0, hole=0, claimed=0, broken=0)
    for index in range(len(test_data)):
        report = conflict_report(semantic_maps[seed_index][index],
                                 instance_lists[seed_index][index])
        for key in add:
            add[key] += report[key]
    conflict_rows.append(add)
    disputed = add["disagree"] + add["only_instance"]
    print("  %5d | %6.2f %% |  %7.2f %% | %5.2f %% | %28.1f %%"
          % (seed, 100 * disputed / pixels_total, 100 * add["overlap"] / pixels_total,
             100 * add["hole"] / pixels_total, 100 * add["broken"] / add["claimed"]))

mean_broken = float(np.mean([100 * r["broken"] / r["claimed"] for r in conflict_rows]))
spread_broken = (max(100 * r["broken"] / r["claimed"] for r in conflict_rows)
                 - min(100 * r["broken"] / r["claimed"] for r in conflict_rows))
print()
print("У середньому по трьох зернах зіпсовано %.1f %% пікселів предметів, розкид %.1f."
      % (mean_broken, spread_broken))
print("Це не помилка якоїсь однієї моделі: кожна відповідає на своє питання,")
print("а питання різні. Саме тому зшивання й доводиться вигадувати.")

## 3 · PQ: одна метрика замість двох

`PQ` (panoptic quality) міряє одну відповідь, у якій кожен піксель має клас і,
якщо це річ, номер предмета. Рахується вона на **сегментах**.

**Сегмент** — це звʼязна відповідь моделі: для речі — маска одного предмета, для
матерії — усі пікселі цього класу на зображенні разом. У нас матерія одна — фон.

Далі три кроки, і кожен із них має просте словесне пояснення.

1. **Зіставлення.** Прогнозований сегмент і істинний сегмент **одного класу**
   зараховуються в пару, якщо їхній `IoU` **більший за 0.5**. Поріг саме такий не
   випадково: при `IoU > 0.5` до одного істинного сегмента не може підійти два
   прогнозовані, тож пари утворюються однозначно, без жодного правила вибору.
2. **RQ** (recognition quality) — це `F1` по сегментах: скільки з них модель
   узагалі знайшла.
3. **SQ** (segmentation quality) — середній `IoU` серед **зарахованих** пар:
   наскільки точні ті маски, які модель таки знайшла.

І добуток: `PQ = SQ × RQ`. Метрика рахується окремо в кожному класі, а потім
усереднюється по класах.

In [ ]:
def mask_iou(first, second):
    """IoU двох масок: спільних пікселів поділити на всі зайняті хоч однією."""
    intersection = np.logical_and(first, second).sum()
    union = np.logical_or(first, second).sum()
    return float(intersection) / float(union) if union else 0.0


def segments_of(class_map, id_map):
    """Розбір panoptic-відповіді на сегменти: матерія цілим, речі поштучно."""
    out = []
    stuff = class_map == 0
    if stuff.any():
        out.append((0, stuff))                  # фон — один сегмент на все зображення
    for instance in np.unique(id_map):
        if instance == 0:
            continue
        mask = id_map == instance
        class_index = int(class_map[mask][0])
        if class_index == 0:
            continue
        out.append((class_index, mask))
    return out


def new_accumulator():
    """Лічильники PQ, спільні на весь набір: PQ рахують не по картинці, а по всіх."""
    return {c: dict(tp=0, fp=0, fn=0, iou=0.0) for c in range(NUM_LABELS)}


def accumulate_pq(accumulator, predicted_segments, true_segments):
    """Зіставляємо сегменти одного класу й додаємо TP, FP, FN до лічильників."""
    for class_index in range(NUM_LABELS):
        predicted = [m for c, m in predicted_segments if c == class_index]
        truth = [m for c, m in true_segments if c == class_index]
        matched_truth = set()
        matched_predicted = 0
        for one in predicted:
            for j, other in enumerate(truth):
                if j in matched_truth:
                    continue
                value = mask_iou(one, other)
                if value > 0.5:                  # строго більше: пара тоді єдина
                    accumulator[class_index]["tp"] += 1
                    accumulator[class_index]["iou"] += value
                    matched_truth.add(j)
                    matched_predicted += 1
                    break
        accumulator[class_index]["fp"] += len(predicted) - matched_predicted
        accumulator[class_index]["fn"] += len(truth) - len(matched_truth)
    return accumulator


def pq_from(accumulator):
    """Із лічильників — PQ, SQ, RQ по класах і в середньому."""
    per_class = {}
    for class_index, cell in accumulator.items():
        denominator = cell["tp"] + 0.5 * cell["fp"] + 0.5 * cell["fn"]
        if denominator == 0:
            continue                             # класу немає ні в істині, ні в прогнозі
        sq = cell["iou"] / cell["tp"] if cell["tp"] else 0.0
        rq = cell["tp"] / denominator
        per_class[class_index] = dict(pq=sq * rq, sq=sq, rq=rq, **cell)
    values = list(per_class.values())
    things = [per_class[c] for c in per_class if c > 0]
    stuff = [per_class[c] for c in per_class if c == 0]
    return dict(pq=float(np.mean([v["pq"] for v in values])),
                sq=float(np.mean([v["sq"] for v in values])),
                rq=float(np.mean([v["rq"] for v in values])),
                pq_things=float(np.mean([v["pq"] for v in things])) if things else 0.0,
                pq_stuff=float(np.mean([v["pq"] for v in stuff])) if stuff else 0.0,
                per_class=per_class)


def truth_panoptic(scene):
    """Еталонна panoptic-розмітка: пара карт «клас» і «номер предмета»."""
    return scene["seg"], scene["ids"]


print("PQ готова. Далі — звірка на прикладі, порахованому руками.")

### Звірка PQ на прикладі, порахованому руками

Полотно 8×8. В істині **дві речі** одного класу й **фон**:

- предмет **A** — рядки 0-1, стовпці 0-3, тобто 8 пікселів;
- предмет **B** — рядки 3-4, стовпці 0-3, теж 8 пікселів;
- фон — решта, 64 − 16 = **48** пікселів.

Модель відповіла так:

- сегмент **A′** — рядки 0-1, стовпці 0-2, тобто 6 пікселів;
- сегменти **B₁** і **B₂** — по 2 пікселі всередині B;
- фон — решта, 64 − 10 = **54** пікселі.

Рахуємо руками.

- `IoU(A′, A) = 6 / 8 = 0.75` — більше за 0.5, пара зарахована;
- `IoU(B₁, B) = 2 / 8 = 0.25` і `IoU(B₂, B) = 0.25` — обидві менші, пари немає;
- отже в класі речей `TP = 1`, `FP = 2`, `FN = 1`;
- `RQ = 1 / (1 + 0.5·2 + 0.5·1) = 1 / 2.5 = 0.4`;
- `SQ = 0.75 / 1 = 0.75`;
- `PQ` речей `= 0.75 × 0.4 = 0.30`.

Тепер фон. Прогнозований фон — це все, крім `A′`, `B₁`, `B₂`. Істинний — усе, крім
`A` і `B`. Спільного в них 48 пікселів (весь істинний фон), разом вони покривають
64 − 10 = 54 пікселі. Отже `IoU = 48 / 54 = 8/9 ≈ 0.8889`, пара одна, `RQ = 1`,
`SQ = PQ = 8/9`.

Середнє по двох класах: `PQ = (0.30 + 8/9) / 2 = 0.5944`.

In [ ]:
hand_class = np.zeros((8, 8), np.int64)
hand_ids = np.zeros((8, 8), np.int64)
hand_class[0:2, 0:4] = 1
hand_ids[0:2, 0:4] = 1                     # предмет A
hand_class[3:5, 0:4] = 1
hand_ids[3:5, 0:4] = 2                     # предмет B

predicted_class = np.zeros((8, 8), np.int64)
predicted_ids = np.zeros((8, 8), np.int64)
predicted_class[0:2, 0:3] = 1
predicted_ids[0:2, 0:3] = 1                # сегмент A′
predicted_class[3, 0:2] = 1
predicted_ids[3, 0:2] = 2                  # сегмент B₁
predicted_class[4, 0:2] = 1
predicted_ids[4, 0:2] = 3                  # сегмент B₂

# ті самі segments_of і accumulate_pq, лише полотно менше
hand_true = [(0, hand_class == 0)] + [(1, hand_ids == k) for k in (1, 2)]
hand_pred = [(0, predicted_class == 0)] + [(1, predicted_ids == k) for k in (1, 2, 3)]
hand_accumulator = {c: dict(tp=0, fp=0, fn=0, iou=0.0) for c in range(2)}
for class_index in range(2):
    predicted = [m for c, m in hand_pred if c == class_index]
    truth = [m for c, m in hand_true if c == class_index]
    matched_truth, matched_predicted = set(), 0
    for one in predicted:
        for j, other in enumerate(truth):
            if j in matched_truth:
                continue
            value = mask_iou(one, other)
            if value > 0.5:
                hand_accumulator[class_index]["tp"] += 1
                hand_accumulator[class_index]["iou"] += value
                matched_truth.add(j)
                matched_predicted += 1
                break
    hand_accumulator[class_index]["fp"] += len(predicted) - matched_predicted
    hand_accumulator[class_index]["fn"] += len(truth) - len(matched_truth)

print("клас речей : TP %d, FP %d, FN %d" % (hand_accumulator[1]["tp"],
                                            hand_accumulator[1]["fp"],
                                            hand_accumulator[1]["fn"]))
things_rq = hand_accumulator[1]["tp"] / (hand_accumulator[1]["tp"]
                                         + 0.5 * hand_accumulator[1]["fp"]
                                         + 0.5 * hand_accumulator[1]["fn"])
things_sq = hand_accumulator[1]["iou"] / hand_accumulator[1]["tp"]
stuff_sq = hand_accumulator[0]["iou"] / hand_accumulator[0]["tp"]
print("речі       : SQ %.4f, RQ %.4f, PQ %.4f  (руками 0.75, 0.4, 0.30)"
      % (things_sq, things_rq, things_sq * things_rq))
print("фон        : SQ %.4f, RQ 1.0000, PQ %.4f  (руками 8/9 = %.4f)"
      % (stuff_sq, stuff_sq, 8 / 9))
hand_pq = (things_sq * things_rq + stuff_sq) / 2
print("разом      : PQ %.4f  (руками %.4f)" % (hand_pq, (0.30 + 8 / 9) / 2))
assert np.isclose(things_sq, 0.75) and np.isclose(things_rq, 0.4), "речі розійшлися!"
assert np.isclose(stuff_sq, 8 / 9), "фон розійшовся!"
assert np.isclose(hand_pq, (0.30 + 8 / 9) / 2), "середнє розійшлося!"
print("✅ наша PQ збігається з підрахунком руками")

### І перевірка самої реалізації

Ідеальний прогноз мусить дати рівно одиницю в усіх трьох множниках. Якщо тут не
одиниця — помилка в коді метрики, а не в моделі.

In [ ]:
perfect = new_accumulator()
for scene in test_data:
    class_map, id_map = truth_panoptic(scene)
    segments = segments_of(class_map, id_map)
    accumulate_pq(perfect, segments, segments)
perfect_result = pq_from(perfect)
print("ідеальна відповідь проти себе: PQ %.4f, SQ %.4f, RQ %.4f"
      % (perfect_result["pq"], perfect_result["sq"], perfect_result["rq"]))
assert np.isclose(perfect_result["pq"], 1.0), "ідеальний прогноз мусить дати одиницю!"
print("✅ реалізація не бреше сама собі")

## 4 · Замір №2: PQ на зшитій відповіді, з розкладом на множники

Щоб порахувати PQ, дві відповіді треба спершу **звести в одну**. Правило зведення
доводиться вигадати — і в цьому вся проблема. Спробуємо три:

- **речі поверх усього.** Маски предметів кладемо за спаданням упевненості; піксель
  дістається тій масці, яка взяла його першою. Усе, що лишилось, — фон;
- **лише там, де семантика згодна.** Маска предмета забирає лише ті пікселі, яким
  семантична модель дала **той самий клас**. Решта відрізається;
- **поріг площі.** Те саме, що перше правило, плюс сегменти, менші за 20 пікселів,
  викидаються.

Жодне з них не випливає з постановки задачі — усі три вигадані нами.

In [ ]:
def merge_two(semantic_map, instances, rule="things", min_area=0):
    """Зшиває карту класів і список предметів в одну panoptic-відповідь."""
    class_map = np.zeros((SIZE, SIZE), np.int64)
    id_map = np.zeros((SIZE, SIZE), np.int64)
    order = np.argsort(-instances["scores"]) if len(instances["masks"]) else []
    next_id = 0
    for k in order:
        mask = instances["masks"][k]
        label = int(instances["labels"][k]) + 1
        if rule == "semantic":
            mask = mask & (semantic_map == label)
        free = mask & (id_map == 0)          # піксель, який ще ніхто не забрав
        if free.sum() < min_area:
            continue
        next_id += 1
        id_map[free] = next_id
        class_map[free] = label
    return class_map, id_map                 # усе, що лишилось, — фон-матерія


def pq_of_maps(maps):
    """PQ по всьому перевірному набору."""
    accumulator = new_accumulator()
    for (class_map, id_map), scene in zip(maps, test_data):
        true_class, true_ids = truth_panoptic(scene)
        accumulate_pq(accumulator, segments_of(class_map, id_map),
                      segments_of(true_class, true_ids))
    return pq_from(accumulator)


merge_results = {}
print("  правило                      |   PQ   |   SQ   |   RQ   | PQ речей | PQ фону")
for rule, min_area, title in (("things", 0, "речі поверх усього"),
                              ("semantic", 0, "де семантика згодна"),
                              ("things", 20, "поріг площі 20 px")):
    rows = []
    for seed_index in range(len(SEEDS)):
        maps = [merge_two(semantic_maps[seed_index][i], instance_lists[seed_index][i],
                          rule, min_area) for i in range(len(test_data))]
        rows.append(pq_of_maps(maps))
    merge_results[title] = rows
    print("  %-28s | %.4f | %.4f | %.4f |  %.4f  | %.4f"
          % (title, float(np.mean([r["pq"] for r in rows])),
             float(np.mean([r["sq"] for r in rows])),
             float(np.mean([r["rq"] for r in rows])),
             float(np.mean([r["pq_things"] for r in rows])),
             float(np.mean([r["pq_stuff"] for r in rows]))))
print()
for title, rows in merge_results.items():
    values = [r["pq"] for r in rows]
    print("  %-28s зерна: %s, розкид %.4f"
          % (title, " ".join("%.4f" % v for v in values), max(values) - min(values)))

## 5 · Замір №3: PQ проти пари mIoU + mask AP

Питання просте: **чи не досить було б рахувати дві звичні метрики окремо?**
`mIoU` міряє пікселі й не бачить предметів, `mask AP` міряє предмети й не бачить
фону. Може, їх пара і є panoptic?

Щоб відповісти, збудуємо шість «моделей» різної якості. Жодну з них не треба
навчати: усі вони зроблені з істини навмисним псуванням, і саме тому ми точно
знаємо, **що саме** в кожній зіпсовано.

In [ ]:
def mean_iou(predicted_class, true_class):
    """mIoU по всіх чотирьох класах, включно з фоном."""
    values = []
    for class_index in range(NUM_LABELS):
        predicted = predicted_class == class_index
        truth = true_class == class_index
        if not truth.any() and not predicted.any():
            continue
        values.append(mask_iou(predicted, truth))
    return float(np.mean(values))


def average_precision(scores, hits, truth_count):
    """Площа під огинальною кривою точність-повнота — та сама, що в темі 23."""
    if truth_count == 0 or len(scores) == 0:
        return 0.0
    order = np.argsort(-np.asarray(scores))
    hits = np.asarray(hits)[order]
    true_positive = np.cumsum(hits)
    precision = true_positive / np.arange(1, len(hits) + 1)
    recall = true_positive / truth_count
    for k in range(len(precision) - 2, -1, -1):
        precision[k] = max(precision[k], precision[k + 1])
    previous, total = 0.0, 0.0
    for k in range(len(recall)):
        total += (recall[k] - previous) * precision[k]
        previous = recall[k]
    return float(total)


def mask_ap50(predictions):
    """mask AP при порозі IoU 0.5, усереднена по класах речей."""
    values = []
    for class_index in range(CLASS_COUNT):
        scores, hits, truth_count = [], [], 0
        for prediction, scene in zip(predictions, test_data):
            chosen = [i for i, c in enumerate(prediction["labels"]) if c == class_index]
            keep = [j for j, c in enumerate(scene["labels"]) if c == class_index]
            truth_count += len(keep)
            taken = set()
            for i in sorted(chosen, key=lambda i: -prediction["scores"][i]):
                best, best_j = 0.0, -1
                for j in keep:
                    if j in taken:
                        continue
                    value = mask_iou(prediction["masks"][i], scene["masks"][j])
                    if value > best:
                        best, best_j = value, j
                scores.append(float(prediction["scores"][i]))
                if best >= 0.5:
                    taken.add(best_j)
                    hits.append(1.0)
                else:
                    hits.append(0.0)
        values.append(average_precision(scores, hits, truth_count))
    return float(np.mean(values))


def panoptic_to_instances(class_map, id_map):
    """Panoptic-відповідь назад у список предметів — щоб порахувати mask AP."""
    masks, labels, scores = [], [], []
    for instance in np.unique(id_map):
        if instance == 0:
            continue
        mask = id_map == instance
        masks.append(mask)
        labels.append(int(class_map[mask][0]) - 1)
        scores.append(float(mask.sum()))     # упевненості нема, тож беремо площу
    return dict(masks=masks, labels=np.array(labels, np.int64),
                scores=np.array(scores, np.float32))


def evaluate_all(maps):
    """Три метрики на одній і тій самій відповіді."""
    accumulator = new_accumulator()
    ious, predictions = [], []
    for (class_map, id_map), scene in zip(maps, test_data):
        true_class, true_ids = truth_panoptic(scene)
        accumulate_pq(accumulator, segments_of(class_map, id_map),
                      segments_of(true_class, true_ids))
        ious.append(mean_iou(class_map, true_class))
        predictions.append(panoptic_to_instances(class_map, id_map))
    result = pq_from(accumulator)
    result["miou"] = float(np.mean(ious))
    result["ap"] = mask_ap50(predictions)
    return result


print("три метрики готові")

In [ ]:
def model_perfect(scene):
    return truth_panoptic(scene)


def model_merged(scene):
    """Пікселі ідеальні, але дотичні предмети свого класу злиплись в один."""
    class_map = scene["seg"]
    id_map = np.zeros((SIZE, SIZE), np.int64)
    next_id = 0
    for class_index in range(1, NUM_LABELS):
        found, marks = cv2.connectedComponents(
            (class_map == class_index).astype(np.uint8), connectivity=8)
        for blob in range(1, found):
            next_id += 1
            id_map[marks == blob] = next_id
    return class_map, id_map


def model_split(scene):
    """Пікселі ідеальні, але кожен предмет розрізано навпіл."""
    class_map = scene["seg"]
    id_map = np.zeros((SIZE, SIZE), np.int64)
    next_id = 0
    for mask in scene["masks"]:
        columns = np.nonzero(mask)[1]
        middle = (columns.min() + columns.max()) // 2
        for part in (mask & (COLUMNS <= middle), mask & (COLUMNS > middle)):
            if part.sum() == 0:
                continue
            next_id += 1
            id_map[part] = next_id
    return class_map, id_map


def model_shifted(scene, pixels=2):
    """Усе правильно, але зсунуто на два пікселі."""
    true_class, true_ids = truth_panoptic(scene)
    class_map = np.zeros((SIZE, SIZE), np.int64)
    id_map = np.zeros((SIZE, SIZE), np.int64)
    class_map[pixels:, pixels:] = true_class[:SIZE - pixels, :SIZE - pixels]
    id_map[pixels:, pixels:] = true_ids[:SIZE - pixels, :SIZE - pixels]
    return class_map, id_map


def model_spoiled_background(scene, share=0.08):
    """Предмети ідеальні, а фон подзьобаний: пікселі дістали клас без предмета."""
    class_map, id_map = truth_panoptic(scene)
    class_map = class_map.copy()
    rng = np.random.default_rng(int(scene["seg"].sum()) % 100003)
    blocks = rng.random((SIZE // 4, SIZE // 4)) < share
    spoiled = np.kron(blocks, np.ones((4, 4), bool)) & (class_map == 0)
    class_map[spoiled] = 1                   # клас кола, але без жодного предмета
    return class_map, id_map


def model_lost(scene, every=3):
    """Кожен третій предмет зникає — його пікселі стають фоном."""
    class_map = np.zeros((SIZE, SIZE), np.int64)
    id_map = np.zeros((SIZE, SIZE), np.int64)
    next_id = 0
    for index, mask in enumerate(scene["masks"]):
        if index % every == 0:
            continue
        next_id += 1
        id_map[mask] = next_id
        class_map[mask] = scene["labels"][index] + 1
    return class_map, id_map


variants = [("ідеальна", model_perfect),
            ("зливає дотичні", model_merged),
            ("ріже предмет навпіл", model_split),
            ("зсув на 2 px", model_shifted),
            ("дзьобає фон", model_spoiled_background),
            ("губить кожен третій", model_lost)]

quality_rows = []
print("  модель                |   PQ   | PQ речей | PQ фону |  mIoU  | mask AP | середнє")
for name, builder in variants:
    result = evaluate_all([builder(scene) for scene in test_data])
    quality_rows.append((name, result))
    print("  %-21s | %.4f |  %.4f  | %.4f | %.4f | %.4f  | %.4f"
          % (name, result["pq"], result["pq_things"], result["pq_stuff"],
             result["miou"], result["ap"], 0.5 * (result["miou"] + result["ap"])))
print()
for key, title in (("pq", "PQ     "), ("miou", "mIoU   "), ("ap", "mask AP")):
    order = sorted(quality_rows, key=lambda row: -row[1][key])
    print("  за %s: %s" % (title, " > ".join(name for name, _ in order)))

### Що з цього видно

Дивись на два рядки.

**«Зливає дотичні»** — це модель, у якої **всі пікселі правильні до одного**. `mIoU`
цього рядка дорівнює одиниці: жодна помилка класу не зроблена. А PQ падає, бо
предмети злились, і сегментів стало менше, ніж є насправді.

**«Дзьобає фон»** — навпаки: усі предмети знайдені ідеально, тож `mask AP` не
помічає нічого. Псується лише фон, до якого `mask AP` байдужий за побудовою.

Отже пара «mIoU + mask AP» має **сліпу пляму з кожного боку**, і жодне усереднення
цих двох чисел її не закриває: у першому рядку `mIoU` каже «ідеально», у другому те
саме каже `mask AP`. PQ — одне число, у якому обидві помилки видно.

## 6 · Замір №4: речі окремо, матерія окремо

`PQ_th` — середнє по класах **речей**, `PQ_st` — по класах **матерії**. У нас
речей три класи, матерії один. Подивимось на них порізно на тій самій зшитій
відповіді, що й у замірі №2.

In [ ]:
class_titles = ["фон (матерія)"] + [name + " (річ)" for name in CLASS_NAMES]
best_maps = [merge_two(semantic_maps[0][i], instance_lists[0][i], "semantic")
             for i in range(len(test_data))]
accumulator = new_accumulator()
for (class_map, id_map), scene in zip(best_maps, test_data):
    true_class, true_ids = truth_panoptic(scene)
    accumulate_pq(accumulator, segments_of(class_map, id_map),
                  segments_of(true_class, true_ids))
detailed = pq_from(accumulator)

print("  клас             |   PQ   |   SQ   |   RQ   |  TP  |  FP  |  FN")
for class_index in range(NUM_LABELS):
    cell = detailed["per_class"][class_index]
    print("  %-16s | %.4f | %.4f | %.4f | %4d | %4d | %4d"
          % (class_titles[class_index], cell["pq"], cell["sq"], cell["rq"],
             cell["tp"], cell["fp"], cell["fn"]))
print()
print("PQ речей   %.4f   (середнє по трьох класах речей)" % detailed["pq_things"])
print("PQ фону    %.4f   (один клас)" % detailed["pq_stuff"])
print("PQ         %.4f   (середнє по всіх чотирьох класах)" % detailed["pq"])
print("наївне середнє по двох категоріях: %.4f"
      % (0.5 * (detailed["pq_things"] + detailed["pq_stuff"])))
print()
print("Сегментів фону рівно стільки, скільки зображень: %d."
      % detailed["per_class"][0]["tp"])
print("Один величезний сегмент на картинку знайти майже неможливо не зуміти —")
print("тому RQ фону тримається біля одиниці, а вся боротьба йде в SQ.")

Звідси головна осторога до `PQ_st`: **матерія майже завжди знаходиться**. Його
сегмент один на зображення й займає більшість пікселів, тож `RQ` матерії
прилипає до одиниці, і `PQ_st` фактично дорівнює його `SQ`.

Через це середнє «(PQ речей + PQ матерії) / 2» лестить моделі: половину оцінки
дає категорія, у якій майже неможливо помилитись у розпізнаванні. Порівнювати
моделі треба або по повному PQ (середнє по **класах**, а не по категоріях), або
дивлячись на `PQ_th` і `PQ_st` окремо — але **ніколи** не на їхнє середнє.

## 7 · Замір №5: власна panoptic-голова

Досі ми зшивали дві моделі. Тепер зберемо **одну**, яка одразу віддає panoptic-карту:

- **семантична гілка** — клас кожного пікселя, чотири канали;
- **гілка голосування** — два числа на піксель: куди від нього до центра свого
  предмета. Пікселі одного предмета показують в одну точку, різних — у різні.

Це рівно та мережа голосування, що в темі 31, тільки використана до кінця:
семантична гілка дає **матерію**, гілка голосування розділяє **речі**, і правило
злиття тут не вигадане, а вбудоване — кожен піксель належить рівно одному згустку
голосів усередині свого класу.

In [ ]:
class PanopticNet(nn.Module):
    """Клас кожного пікселя і зсув від пікселя до центра його предмета."""

    def __init__(self, width=PAN_WIDTH):
        super().__init__()
        self.body = UNetBody(1, width)
        self.semantic = nn.Conv2d(width, NUM_LABELS, 1)
        self.offset = nn.Conv2d(width, 2, 1)

    def forward(self, x):
        features = self.body(x)
        return self.semantic(features), self.offset(features)


def pack_votes(data):
    """Ціль зсуву: від кожного пікселя предмета до центра мас цього предмета."""
    images = torch.from_numpy(np.stack([s["image"] for s in data])[:, None])
    segmentation = torch.from_numpy(np.stack([s["seg"] for s in data]))
    offsets = np.zeros((len(data), 2, SIZE, SIZE), np.float32)
    for index, scene in enumerate(data):
        for mask in scene["masks"]:
            rows, columns = np.nonzero(mask)
            offsets[index, 0][mask] = columns.mean() - COLUMNS[mask]
            offsets[index, 1][mask] = rows.mean() - ROWS[mask]
    return images, segmentation, torch.from_numpy(offsets)


def train_panoptic(images, segmentation, offsets, seed, epochs=PAN_EPOCHS,
                   batch_size=8, learning_rate=3e-3):
    torch.manual_seed(seed)
    model = PanopticNet()
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    started = time.time()
    model.train()
    for _epoch in range(epochs):
        order = torch.randperm(len(images))
        for start in range(0, len(images), batch_size):
            batch = order[start:start + batch_size]
            logits, predicted_offsets = model(images[batch])
            loss = F.cross_entropy(logits, segmentation[batch])
            foreground = segmentation[batch] > 0
            if foreground.any():
                weight = foreground.unsqueeze(1).float()
                # зсуви вчимо тільки на пікселях предметів: у фону центра немає
                loss = loss + (F.l1_loss(predicted_offsets, offsets[batch],
                                         reduction="none") * weight).sum() \
                    / weight.sum().clamp(min=1) / 8.0
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    return model, time.time() - started


def find_peaks(votes_x, votes_y, min_votes=25):
    """Локальні максимуми в сітці голосів — це й будуть центри предметів."""
    accumulator = np.zeros((SIZE, SIZE), np.float32)
    columns = np.clip(np.round(votes_x).astype(int), 0, SIZE - 1)
    rows = np.clip(np.round(votes_y).astype(int), 0, SIZE - 1)
    np.add.at(accumulator, (rows, columns), 1.0)
    smooth = F.avg_pool2d(torch.from_numpy(accumulator)[None, None], 3, 1, 1)
    highest = F.max_pool2d(smooth, 5, 1, 2)
    is_peak = ((smooth == highest) & (smooth * 9 >= min_votes))[0, 0].numpy()
    return list(zip(*np.nonzero(is_peak)))


@torch.no_grad()
def panoptic_scene(model, image, min_votes=25):
    """Одна відповідь: клас кожного пікселя і номер предмета там, де це річ."""
    model.eval()
    logits, offsets = model(torch.from_numpy(image)[None, None])
    labels = logits[0].argmax(dim=0).numpy()
    offsets = offsets[0].numpy()
    class_map = labels.copy()
    id_map = np.zeros((SIZE, SIZE), np.int64)
    next_id = 0
    for class_index in range(1, NUM_LABELS):
        selected = labels == class_index
        if selected.sum() < 8:
            class_map[selected] = 0            # надто дрібне — оголошуємо фоном
            continue
        votes_x = COLUMNS[selected] + offsets[0][selected]
        votes_y = ROWS[selected] + offsets[1][selected]
        peaks = find_peaks(votes_x, votes_y, min_votes)
        if not peaks:
            peaks = [(int(round(votes_y.mean())), int(round(votes_x.mean())))]
        peak_y = np.array([p[0] for p in peaks], np.float32)
        peak_x = np.array([p[1] for p in peaks], np.float32)
        distance = (votes_x[:, None] - peak_x[None, :]) ** 2 \
            + (votes_y[:, None] - peak_y[None, :]) ** 2
        owner = distance.argmin(axis=1)
        rows, columns = np.nonzero(selected)
        for peak_index in range(len(peaks)):
            chosen = owner == peak_index
            if chosen.sum() < 8:               # згусток із кількох пікселів — це шум
                class_map[rows[chosen], columns[chosen]] = 0
                continue
            next_id += 1
            id_map[rows[chosen], columns[chosen]] = next_id
    return class_map, id_map


print("параметрів у panoptic-голові: %d" % sum(p.numel() for p in PanopticNet().parameters()))

In [ ]:
vote_images, vote_segmentation, vote_offsets = pack_votes(train_data)
own_rows, own_seconds = [], []
for seed in SEEDS:
    model, elapsed = train_panoptic(vote_images, vote_segmentation, vote_offsets, seed)
    maps = [panoptic_scene(model, scene["image"]) for scene in test_data]
    result = evaluate_all(maps)
    own_rows.append(result)
    own_seconds.append(elapsed)
    if seed == SEEDS[0]:
        own_model, own_maps = model, maps
    print("  зерно %d (%3.0f с): PQ %.4f = SQ %.4f × RQ %.4f | речі %.4f фон %.4f"
          % (seed, elapsed, result["pq"], result["sq"], result["rq"],
             result["pq_things"], result["pq_stuff"]))
own_pq = [r["pq"] for r in own_rows]
print("  середнє: PQ %.4f, розкид %.4f" % (float(np.mean(own_pq)),
                                           max(own_pq) - min(own_pq)))
print()

stitched = merge_results["де семантика згодна"]
stitched_pq = [r["pq"] for r in stitched]
print("  зшивання двох моделей (найкраще правило): PQ %.4f, розкид %.4f"
      % (float(np.mean(stitched_pq)), max(stitched_pq) - min(stitched_pq)))
print("  власна panoptic-голова                  : PQ %.4f, розкид %.4f"
      % (float(np.mean(own_pq)), max(own_pq) - min(own_pq)))
difference = float(np.mean(own_pq)) - float(np.mean(stitched_pq))
print("  різниця %+.4f при найбільшому розкиді %.4f"
      % (difference, max(max(own_pq) - min(own_pq), max(stitched_pq) - min(stitched_pq))))

### Як це виглядає

Одна сцена, розібрана трьома способами: істина, зшивання двох моделей і власна
голова. Кольори — окремі предмети, сірий — фон-матерія.

In [ ]:
scene_index = 3
scene = test_data[scene_index]
stitched_map = merge_two(semantic_maps[0][scene_index], instance_lists[0][scene_index],
                         "semantic")
figure, axes = plt.subplots(1, 4, figsize=(13, 3.4))
panels = [("сцена", None), ("істина", truth_panoptic(scene)),
          ("зшито з двох", stitched_map), ("власна голова", own_maps[scene_index])]
for axis, (title, maps) in zip(axes, panels):
    if maps is None:
        axis.imshow(scene["image"], cmap="gray", vmin=0, vmax=1)
    else:
        axis.imshow(maps[1], cmap="tab10", vmin=0, vmax=9)
        title = "%s: %d" % (title, len(np.unique(maps[1])) - 1)
    axis.set_title(title, fontsize=9)
    axis.axis("off")
plt.tight_layout()
plt.show()
print("на цій сцені предметів %d" % len(scene["masks"]))

## 8 · Сегментація за підказкою: інша постановка

Усе попереднє — це «сегментуй усе, що знаєш». Модель мусить знати список класів
наперед, і про предмет поза списком їй сказати нічого.

SAM ставить питання інакше: **«сегментуй те, на що я показав»**. Модель не
класифікує нічого. Вона приймає зображення й **підказку** — точку, рамку, грубу
маску — і віддає маску предмета, на який ця підказка вказує. Це та сама зміна, що
CLIP зробив для класифікації ([тема 22](../22-clip/lecture.html)): замість
фіксованого списку відповідей — довільний запит.

Ваг справжнього SAM у нас немає, тож ми навчимо власну крихітну мережу за тією
самою постановкою. Вхід — **два канали**: зображення й карта підказки. Вихід —
маска. Жодного класу на виході немає, і це не спрощення, а точна копія постановки.

In [ ]:
DISC = 2                        # радіус кружечка, яким малюємо точку-підказку
PROMPT_SCENES = 90              # менше сцен, ніж у решті зошита: підказок і так по кілька на сцену


def point_inside(mask):
    """Піксель предмета, найближчий до його центра мас — стабільна підказка."""
    rows, columns = np.nonzero(mask)
    distance = (rows - rows.mean()) ** 2 + (columns - columns.mean()) ** 2
    best = int(np.argmin(distance))
    return int(rows[best]), int(columns[best])


def prompt_point(row, column):
    field = np.zeros((SIZE, SIZE), np.float32)
    field[(ROWS - row) ** 2 + (COLUMNS - column) ** 2 <= DISC * DISC] = 1.0
    return field


def prompt_box(box):
    field = np.zeros((SIZE, SIZE), np.float32)
    x0, y0, x1, y1 = [int(v) for v in box]
    field[y0:y1, x0:x1] = 1.0
    return field


def build_prompts(data):
    """Для кожного предмета: підказка й три однаково чесні відповіді на неї."""
    out = []
    for scene in data:
        foreground = (scene["seg"] > 0).astype(np.uint8)
        _found, groups = cv2.connectedComponents(foreground, connectivity=8)
        for index, mask in enumerate(scene["masks"]):
            row, column = point_inside(mask)
            group = groups == groups[row, column]        # уся злипла група
            same_class = group & (scene["seg"] == scene["labels"][index] + 1)
            out.append(dict(image=scene["image"], row=row, column=column,
                            box=scene["boxes"][index],
                            levels=[mask, same_class, group]))
    return out


def stack_prompts(prompts, levels, kind="point"):
    """Тензори для навчання: канал зображення, канал підказки, ціль."""
    inputs, targets = [], []
    for prompt in prompts:
        hint = prompt_point(prompt["row"], prompt["column"]) if kind == "point" \
            else prompt_box(prompt["box"])
        for level in levels:
            inputs.append(np.stack([prompt["image"], hint]))
            targets.append(prompt["levels"][level].astype(np.float32))
    return torch.from_numpy(np.stack(inputs)), torch.from_numpy(np.stack(targets))


train_prompts = build_prompts(train_data[:PROMPT_SCENES])
test_prompts = build_prompts(test_data)
ambiguous = [not np.array_equal(p["levels"][0], p["levels"][2]) for p in test_prompts]
print("підказок: навчальних %d, перевірних %d" % (len(train_prompts), len(test_prompts)))
print("з них неоднозначних (предмет ≠ група): %d, тобто %.1f %%"
      % (sum(ambiguous), 100 * sum(ambiguous) / len(test_prompts)))

## 9 · Замір №6: своя модель за підказкою

Мережа — те саме тіло U-Net, що й раніше, тільки вхідних каналів два й вихід один.
Ціль — маска **того предмета**, у який показує точка.

Порівняємо її з іншим способом дати відповідь на ту саму підказку: **сегментувати
все, а потім вибрати**. Найчесніший представник цього способу — семантична модель
плюс звʼязні плями з [теми 31](../31-instance-segmentation/lecture.html): беремо
пляму, у яку потрапила точка.

In [ ]:
class PromptNet(nn.Module):
    """Приймає зображення й підказку, віддає `outputs` масок-кандидатів."""

    def __init__(self, width=PROMPT_WIDTH, outputs=1):
        super().__init__()
        self.body = UNetBody(2, width)
        # у кожного виходу власна голова: з одним шаром 1×1 три маски виходять
        # лінійними комбінаціями тих самих ознак і злипаються в одну
        self.heads = nn.ModuleList([
            nn.Sequential(nn.Conv2d(width, width, 3, padding=1),
                          nn.BatchNorm2d(width), nn.ReLU(),
                          nn.Conv2d(width, 1, 1))
            for _ in range(outputs)])
        # різні початкові зсуви розводять виходи: один схильний казати «мало»,
        # інший «багато». Без цього перемагає один вихід, а решта лишаються
        # сирими — і три маски вироджуються в одну
        starts = [0.0] if outputs == 1 else np.linspace(-3.0, 3.0, outputs)
        for head, start in zip(self.heads, starts):
            nn.init.constant_(head[-1].bias, float(start))
        self.outputs = outputs

    def forward(self, x):
        features = self.body(x)
        return torch.cat([head(features) for head in self.heads], dim=1)


def train_prompt(inputs, targets, seed, outputs=1, epochs=PROMPT_EPOCHS,
                 batch_size=16, learning_rate=3e-3):
    torch.manual_seed(seed)
    model = PromptNet(outputs=outputs)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    started = time.time()
    model.train()
    for _epoch in range(epochs):
        order = torch.randperm(len(inputs))
        for start in range(0, len(inputs), batch_size):
            batch = order[start:start + batch_size]
            logits = model(inputs[batch])
            truth = targets[batch]
            if outputs == 1:
                loss = F.binary_cross_entropy_with_logits(logits[:, 0], truth)
            else:
                # правило SAM: учиться лише той вихід, який цього разу вгадав найкраще
                each = F.binary_cross_entropy_with_logits(
                    logits, truth.unsqueeze(1).expand_as(logits),
                    reduction="none").mean(dim=(2, 3))
                loss = each.min(dim=1).values.mean()
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    return model, time.time() - started


@torch.no_grad()
def prompt_iou(model, inputs, targets, batch_size=32):
    """IoU найкращого з виходів моделі — протокол самого SAM."""
    model.eval()
    values, chosen_output = [], []
    for start in range(0, len(inputs), batch_size):
        logits = model(inputs[start:start + batch_size])
        predicted = (torch.sigmoid(logits) >= 0.5).numpy()
        truth = targets[start:start + batch_size].numpy() >= 0.5
        for k in range(len(truth)):
            best, best_index = 0.0, 0
            for channel in range(predicted.shape[1]):
                value = mask_iou(predicted[k, channel], truth[k])
                if value > best:
                    best, best_index = value, channel
            values.append(best)
            chosen_output.append(best_index)
    return np.array(values), np.array(chosen_output)


point_train_x, point_train_y = stack_prompts(train_prompts, [0], "point")
point_test_x, point_test_y = stack_prompts(test_prompts, [0], "point")
print("навчальних прикладів: %d" % len(point_train_x))
print("параметрів у мережі за підказкою: %d" % sum(p.numel() for p in PromptNet().parameters()))

In [ ]:
point_scores, point_seconds, point_values = [], [], []
for seed in SEEDS:
    model, elapsed = train_prompt(point_train_x, point_train_y, seed)
    values, _chosen = prompt_iou(model, point_test_x, point_test_y)
    point_scores.append(float(values.mean()))
    point_values.append(values)          # знадобляться в замірі №8, щоб не вчити вдруге
    point_seconds.append(elapsed)
    if seed == SEEDS[0]:
        point_model = model
    print("  зерно %d (%2.0f с): IoU %.4f" % (seed, elapsed, point_scores[-1]))
print("  середнє: IoU %.4f, розкид %.4f"
      % (float(np.mean(point_scores)), max(point_scores) - min(point_scores)))
print()

# спосіб «сегментуй усе, потім вибери»: пляма семантичної маски під точкою
blob_values = []
offset = 0
for scene_index, scene in enumerate(test_data):
    predicted_semantic = semantic_maps[0][scene_index]
    for _ in range(len(scene["masks"])):
        prompt = test_prompts[offset]
        offset += 1
        class_index = predicted_semantic[prompt["row"], prompt["column"]]
        if class_index == 0:
            blob_values.append(0.0)            # модель вважає цю точку фоном
            continue
        _found, marks = cv2.connectedComponents(
            (predicted_semantic == class_index).astype(np.uint8), connectivity=8)
        blob = marks == marks[prompt["row"], prompt["column"]]
        blob_values.append(mask_iou(blob, prompt["levels"][0]))
print("  «сегментуй усе, потім вибери пляму під точкою»: IoU %.4f"
      % float(np.mean(blob_values)))
print()
print("Різниця %+.4f. Той спосіб не має жодного механізму, щоб відрізнити"
      % (float(np.mean(point_scores)) - float(np.mean(blob_values))))
print("предмет від сусіда свого класу: він віддає пляму цілком.")

## 10 · Замір №7: одна маска на неоднозначну підказку

Точка не визначає відповідь однозначно. Показавши в коло, яке торкається квадрата,
ти міг мати на увазі:

1. **саме це коло** — окремий предмет;
2. **усі кола цієї злиплої групи** — свій клас у групі;
3. **усю групу цілком** — все, що злиплось в одну пляму.

Усі три відповіді чесні. У нас таких підказок більшість — точне число надрукувала
клітинка вище.

Справжній SAM віддає **три маски** на одну підказку саме тому. І вчиться він так:
похибка рахується для кожного виходу, а назад іде **лише найменша** — тобто
навчається тільки той вихід, який цього разу вгадав найкраще.

Порівнюємо чесно: та сама мережа, ті самі дані, той самий бюджет навчання. Різниця
лише в кількості виходів. Оцінка теж одна для обох: `IoU` **найкращого** з виходів
— так міряють SAM.

In [ ]:
ambiguous_train_x, ambiguous_train_y = stack_prompts(train_prompts[:180], [0, 1, 2], "point")
ambiguous_test_x, ambiguous_test_y = stack_prompts(test_prompts, [0, 1, 2], "point")
print("навчальних прикладів: %d (та сама підказка з трьома різними цілями)"
      % len(ambiguous_train_x))

ambiguity_results = {}
for outputs in (1, 3):
    scores, seconds, spreads = [], [], []
    for seed in SEEDS:
        model, elapsed = train_prompt(ambiguous_train_x, ambiguous_train_y, seed,
                                      outputs=outputs, epochs=AMBIGUOUS_EPOCHS)
        values, chosen = prompt_iou(model, ambiguous_test_x, ambiguous_test_y)
        scores.append(float(values.mean()))
        seconds.append(elapsed)
        spreads.append(np.bincount(chosen, minlength=3))
        if outputs == 3 and seed == SEEDS[0]:
            three_model = model
        if outputs == 1 and seed == SEEDS[0]:
            one_model = model
    ambiguity_results[outputs] = scores
    print("  виходів %d (%2.0f с на зерно): IoU %.4f, розкид %.4f"
          % (outputs, float(np.mean(seconds)), float(np.mean(scores)),
             max(scores) - min(scores)))
    if outputs == 3:
        print("      скільки разів вигравав кожен вихід, по зернах:")
        for seed, counts in zip(SEEDS, spreads):
            print("        зерно %d: %s" % (seed, list(counts)))

gain = float(np.mean(ambiguity_results[3])) - float(np.mean(ambiguity_results[1]))
spread_one = max(ambiguity_results[1]) - min(ambiguity_results[1])
spread_three = max(ambiguity_results[3]) - min(ambiguity_results[3])
print()
print("Три маски проти однієї: %+.4f" % gain)
print("Розкид по зернах: один вихід %.4f, три виходи %.4f — різниця в %.0f разів."
      % (spread_one, spread_three, spread_one / max(spread_three, 1e-6)))
print()
print("Друга половина результату важливіша за першу. Модель з одним виходом дістає")
print("суперечливі градієнти: той самий вхід тягне її то до предмета, то до групи.")
print("Куди вона осяде, вирішує зерно — звідси розкид. Три виходи цієї суперечності")
print("не мають: кожен рівень дістається своєму виходу, і числа перестають стрибати.")

## 11 · Замір №8: підказка рамкою проти підказки точкою

Точка каже лише «десь тут». Рамка каже ще й «і ось до цих меж» — тобто сама по
собі знімає ту неоднозначність, через яку потрібні три маски.

Мережа, дані й бюджет ті самі; змінюється лише те, що намальовано в другому каналі:
кружечок радіуса 2 або залитий прямокутник. Ціль в обох випадках одна — **предмет**.

Дивимось не лише на середнє, а й окремо на дві частини перевірки: підказки, де
предмет ні з ким не злипся (там неоднозначності немає), і решту.

In [ ]:
box_train_x, box_train_y = stack_prompts(train_prompts, [0], "box")
box_test_x, box_test_y = stack_prompts(test_prompts, [0], "box")
ambiguous_flags = np.array(ambiguous)

box_scores, box_seconds = [], []
box_split, point_split = [], []
for seed in SEEDS:
    model, elapsed = train_prompt(box_train_x, box_train_y, seed)
    values, _chosen = prompt_iou(model, box_test_x, box_test_y)
    box_scores.append(float(values.mean()))
    box_seconds.append(elapsed)
    box_split.append((float(values[~ambiguous_flags].mean()),
                      float(values[ambiguous_flags].mean())))
    print("  рамка, зерно %d (%2.0f с): IoU %.4f" % (seed, elapsed, box_scores[-1]))

# моделі за точкою вже навчені в замірі №6 — просто дивимось на ті самі числа
# в розрізі неоднозначності, замість того щоб учити їх удруге
for values in point_values:
    point_split.append((float(values[~ambiguous_flags].mean()),
                        float(values[ambiguous_flags].mean())))

print()
print("  підказка | усі підказки | предмет сам | предмет у групі")
print("  точка    |    %.4f    |   %.4f    |     %.4f"
      % (float(np.mean(point_scores)),
         float(np.mean([s[0] for s in point_split])),
         float(np.mean([s[1] for s in point_split]))))
print("  рамка    |    %.4f    |   %.4f    |     %.4f"
      % (float(np.mean(box_scores)),
         float(np.mean([s[0] for s in box_split])),
         float(np.mean([s[1] for s in box_split]))))
print()
print("  розкид по зернах: точка %.4f, рамка %.4f"
      % (max(point_scores) - min(point_scores), max(box_scores) - min(box_scores)))
print("  різниця в середньому: %+.4f"
      % (float(np.mean(box_scores)) - float(np.mean(point_scores))))

### Як це виглядає

Одна сцена, одна точка, і три однаково чесні відповіді на неї. Праворуч — що
насправді видала мережа з одним виходом і мережа з трьома.

In [ ]:
example = None
for prompt_index, prompt in enumerate(test_prompts):
    if ambiguous[prompt_index]:
        example = (prompt_index, prompt)
        break

if example is None:
    print("на цій перевірці неоднозначних підказок не знайшлося")
else:
    prompt_index, prompt = example
    hint = prompt_point(prompt["row"], prompt["column"])
    single = torch.from_numpy(np.stack([prompt["image"], hint])[None])
    with torch.no_grad():
        one_mask = (torch.sigmoid(one_model(single))[0, 0].numpy() >= 0.5)
        three_masks = (torch.sigmoid(three_model(single))[0].numpy() >= 0.5)
    titles = ["сцена й точка", "1 · предмет", "2 · свій клас у групі", "3 · уся група",
              "одна маска", "три маски: краща"]
    best_channel = int(np.argmax([mask_iou(three_masks[c], prompt["levels"][0])
                                  for c in range(three_masks.shape[0])]))
    pictures = [None, prompt["levels"][0], prompt["levels"][1], prompt["levels"][2],
                one_mask, three_masks[best_channel]]
    figure, axes = plt.subplots(1, 6, figsize=(15, 2.9))
    for axis, title, picture in zip(axes, titles, pictures):
        if picture is None:
            axis.imshow(prompt["image"], cmap="gray", vmin=0, vmax=1)
        else:
            axis.imshow(picture, cmap="gray", vmin=0, vmax=1)
        axis.plot([prompt["column"]], [prompt["row"]], "o", color="#c2185b", markersize=5)
        axis.set_title(title, fontsize=8)
        axis.axis("off")
    plt.tight_layout()
    plt.show()
    print("IoU однієї маски з предметом %.4f, зі своїм класом у групі %.4f, з групою %.4f"
          % (mask_iou(one_mask, prompt["levels"][0]),
             mask_iou(one_mask, prompt["levels"][1]),
             mask_iou(one_mask, prompt["levels"][2])))

In [ ]:
print("час зошита наскрізь: %.0f с" % (time.time() - NOTEBOOK_STARTED))
print()
print("з них на навчання:")
print("  семантичні мережі  %4.0f с" % sum(semantic_time))
print("  instance-мережі    %4.0f с" % sum(instance_time))
print("  panoptic-голови    %4.0f с" % sum(own_seconds))
print("  мережі за підказкою %3.0f с" % (sum(point_seconds) + sum(box_seconds)))

## Завдання

**🟢 Рівень 1.** Прожени замір №1 при `max_objects` = 2, 3, 4, 6 і побудуй криву:
частка зіпсованих пікселів залежно від щільності сцени. Де вона переходить за 20 %?

**🟡 Рівень 2.** Додай четверте правило злиття — «пріоритет упевненості»: сортуй
разом і маски предметів, і ймовірності семантичної моделі, і клади в одну карту за
спаданням. Порівняй його PQ з трьома нашими на трьох зернах.

**🔴 Рівень 3.** Напиши PQ з нуля, звір її на прикладі, порахованому руками (він у
розділі 3), і порівняй **два способи злиття** на власних сценах: додай у
`shape_mask` кільце й хрест, щоб зʼявились неопуклі предмети. Чи збережеться
перевага правила «де семантика згодна»? Три зерна на кожен спосіб.

Повний опис — у [homework.html](homework.html).